<div style="display: flex; align-items: center; justify-content: center; gap: 20px;">
  <!-- Left Column: Image -->
  <div style="flex: 1; text-align: center;">
    <img src="https://i0.wp.com/cdcs.web.ua.pt/wp-content/uploads/2022/05/cropped-cropped-Picture13-1.png?w=968" width="370" height="200" style="display: block; margin: auto;"/>
  </div>

  <!-- Right Column: Text -->
  <div style="flex: 1; text-align: left;">
    <div><strong style="color: #4F5B63; font-size: 1.5em;">Master in Data Science for Social Sciences</strong></div>
    <div><strong style="color: #4F5B63; font-size: 1.2em;">University of Aveiro</strong></div>
    <p style="color: #46627F; font-weight: bold; font-style: italic;">Introduction to Data Science - 2024/2025</p>
    <p style="color: #4F5B63;">João Lourenço Marques</p>
    <p style="color: #4F5B63;">Paulo Batista</p>
  </div>
</div>


<div style="display: flex; justify-content: space-around; align-items: flex-start;">
  <div style="width: 100%; padding: 10px; box-shadow: 0 2px 4px rgba(0,0,0,0.1); margin: 10px;">
    <h2><h1 style="text-align: center; font-size: 4em; color: #46627F; margin-top: 0; margin-bottom: 0; line-height: 1;">Bibliometric Analysis</h1>
<h1 style="text-align: center; color: #B1C0CF; margin-top: 0; margin-bottom: 0; line-height: 1;"> -Scopus Dataset- </h1></h2>
      </div>
</div>

In [1]:
import requests
import pandas as pd
import time
import matplotlib.pyplot as plt

import os
import sys
from decouple import Config, RepositoryEnv


import math

In [2]:
# Set up project root as working directory

from pathlib import Path

# Find the project root (assuming marker-based or script-relative path)
def find_project_root(marker="README.md"):
    current_dir = Path.cwd()
    while current_dir != current_dir.parent:  # Traverse up until root
        if (current_dir / marker).exists():
            return current_dir
        current_dir = current_dir.parent
    raise FileNotFoundError(f"Marker '{marker}' not found in any parent directory.")

project_root = find_project_root()
sys.path.append(str(project_root)) 

# Or use a relative path: project_root = Path(__file__).resolve().parent.parent
os.chdir(project_root)
print(f"Working directory set to: {project_root}")

Working directory set to: c:\Users\paulo\OneDrive\TRABALHO_AULAS\AL20252026\1S_ICD\TP\projICD


## Configuração Chave API

You can get your API key by creating an account at https://dev.elsevier.com/apikey/manage and generating a new key.

In [3]:
from decouple import config

# https://medium.com/@alensabu12xtz/configure-environment-variables-with-python-decouple-b5f5446381a3

MY_SCOPUS_API_KEY = config('MY_SCOPUS_API_KEY')

# print('my scopus api key is :', MY_SCOPUS_API_KEY) 

In [4]:
from decouple import Config, RepositoryEnv

DOTENV_FILE = './.env'
env_config = Config(RepositoryEnv(DOTENV_FILE))



# use the Config().get() method as you normally would since 
# decouple.config uses that internally. 
# i.e. config('SECRET_KEY') = env_config.get('SECRET_KEY')
MY_SCOPUS_API_KEY = env_config.get('MY_SCOPUS_API_KEY')

# print('my scopus api key is :', MY_SCOPUS_API_KEY) 

# Download data from Scopus API using your API key stored in a .env file

In [5]:
# Defina a sua chave de API da Elsevier
api_key = MY_SCOPUS_API_KEY
# query = "data science AND social science" 
query = ' "Smart Cities" AND "Open Data Infrastructure" OR "Government Open Data" OR "Administrative Open Data" ' # Tema dos artigos
url = "https://api.elsevier.com/content/search/scopus"

## First run on Scopus API to get search results (identify total number of records)

In [6]:
# IMPORTANTE: https://dev.elsevier.com/sc_search_tips.html


responseScopus = requests.get("https://api.elsevier.com/content/search/scopus",
                    headers={'Accept':'application/json',
                             'X-ELS-APIKey': MY_SCOPUS_API_KEY},
                         
                    params={    'query' : query,
                                #'start' : "1"
                        }

                        )

print(responseScopus.url)

dataScopus_json = responseScopus.json() 



https://api.elsevier.com/content/search/scopus?query=+%22Smart+Cities%22+AND+%22Open+Data+Infrastructure%22+OR+%22Government+Open+Data%22+OR+%22Administrative+Open+Data%22+


In [7]:
# Visualize first record
dataScopus_json.get("search-results", {}).get("entry")[0]

{'@_fa': 'true',
 'link': [{'@_fa': 'true',
   '@ref': 'self',
   '@href': 'https://api.elsevier.com/content/abstract/scopus_id/105012602138'},
  {'@_fa': 'true',
   '@ref': 'author-affiliation',
   '@href': 'https://api.elsevier.com/content/abstract/scopus_id/105012602138?field=author,affiliation'},
  {'@_fa': 'true',
   '@ref': 'scopus',
   '@href': 'https://www.scopus.com/inward/record.uri?partnerID=HzOxMe3b&scp=105012602138&origin=inward'},
  {'@_fa': 'true',
   '@ref': 'scopus-citedby',
   '@href': 'https://www.scopus.com/inward/citedby.uri?partnerID=HzOxMe3b&scp=105012602138&origin=inward'}],
 'prism:url': 'https://api.elsevier.com/content/abstract/scopus_id/105012602138',
 'dc:identifier': 'SCOPUS_ID:105012602138',
 'eid': '2-s2.0-105012602138',
 'dc:title': 'Digital Twin or Digital Kin: Misunderstandings and Myths about Urban Simulation, and Directions for Change',
 'dc:creator': 'Payne W.B.',
 'prism:publicationName': 'Journal of Planning Education and Research',
 'prism:issn'

In [8]:
total_results = dataScopus_json.get("search-results", {}).get("opensearch:totalResults")
print("Total Results:", total_results)

Total Results: 265


In [9]:
# Campos a extrair
fields = ",".join([
    "dc:title", "dc:creator", "prism:publicationName", "prism:doi", "affiliation", "citedby-count",
    "pubmed-id", "issn", "isbn", "abstract", "author-keywords", "index-keywords", "publisher",
    "funding-text", "conference-name", "conference-date", "language", "source", "eid"
])

The following script block includes a get_abstract() function, to connect to the Abstract Retrieval API endpoint to fetch the full abstract for each article using its EID (Elsevier ID) obtained from the initial search results / search endpoint.

Thus, this is a two-step retrieval process:

First, the Search API (https://dev.elsevier.com/documentation/ScopusSearchAPI.wadl) retrieves basic metadata
Then, for each article, the Abstract Retrieval API (https://dev.elsevier.com/documentation/AbstractRetrievalAPI.wadl) fetches the full abstract
Rate limiting: Added a 0.5-second delay between abstract retrievals to respect API rate limits while still being efficient.

Progress tracking: Added print statements so you can monitor the collection progress.

Error handling: The get_abstract() function includes try-except to handle potential errors gracefully.

In [ ]:
# Function to retrieve abstract using Abstract Retrieval API
def get_abstract(eid, api_key):
    """
    Retrieve abstract for a given EID using the Abstract Retrieval API
    """
    if not eid:
        return None
    
    abstract_url = f"https://api.elsevier.com/content/abstract/eid/{eid}"
    headers = {
        'Accept': 'application/json',
        'X-ELS-APIKey': api_key
    }
    
    try:
        response = requests.get(abstract_url, headers=headers)
        if response.status_code == 200:
            data = response.json()
            # Navigate through the JSON structure to find the abstract
            abstract_data = data.get('abstracts-retrieval-response', {})
            coredata = abstract_data.get('coredata', {})
            abstract_text = coredata.get('dc:description', None)
            return abstract_text
        else:
            return None
    except Exception as e:
        print(f"Error retrieving abstract for EID {eid}: {e}")
        return None

# Inicializar variáveis para paginação e armazenamento de resultados
start = 0
all_articles = []
max_results_per_request = 25  # Limite máximo da API por requisição
request_limit = 20000  # Rate limits: 20,000 requests per week for Scopus API see on: https://dev.elsevier.com/api_key_settings.html
cursor = "*"

# Parâmetros da requisição
params = {
    "query": query,
    "apiKey": api_key,
    "count": 25,  # Número máximo de artigos por requisição
    "start": 0,
    "field": fields,  # Especificando os campos
    "cursor" : cursor
}

print(f"Starting data collection for {total_results} articles...")

for i in range(0, math.ceil(int(total_results) / int(max_results_per_request))):
    
    print(f"Fetching batch {i+1}/{math.ceil(int(total_results) / int(max_results_per_request))}...")
    
    # Envio da requisição GET
    response = requests.get(url, params=params)
    data = response.json()  # Dados em formato JSON

    # Verificar e processar os resultados
    if response.status_code == 200:
        articles = data.get("search-results", {}).get("entry", [])
        
        # Extrair dados e armazenar em um DataFrame
        articles_data = []
        for article in articles:
            eid = article.get("eid")
            
            # Retrieve abstract using Abstract Retrieval API
            abstract = get_abstract(eid, api_key)
            time.sleep(0.5)  # Small delay to respect rate limits
            
            article_info = {
                "Title": article.get("dc:title"),
                "Authors": article.get("dc:creator"),
                "Journal": article.get("prism:publicationName"),
                "DOI": article.get("prism:doi"),
                "Cited by": article.get("citedby-count"),
                "Affiliations": article.get("affiliation"),
                "Abstract": abstract,  # Now using the retrieved abstract
                "Keywords": article.get("authkeywords"),
                "Index Keywords": article.get("index-keywords"),
                "Publisher": article.get("publisher"),
                "Conference Name": article.get("conference-name"),
                "Conference Date": article.get("conference-date"),
                "Language": article.get("language"),
                "Source": article.get("source"),
                "EID": eid,
                "ISSN": article.get("issn"),
                "ISBN": article.get("isbn"),
                "PubMed ID": article.get("pubmed-id"),
                "Funding Text": article.get("funding-text")
            }
            articles_data.append(article_info)
        
        wait_time = 1  # Tempo de espera em segundos
        time.sleep(wait_time)  # Pausa para evitar exceder o limite de requisição
        all_articles.extend(articles_data)
        cursor = data["search-results"]["cursor"]['@next']
        params.update(cursor=cursor)
        
        print(f"Collected {len(all_articles)} articles so far...")

print(f"\nData collection complete! Total articles: {len(all_articles)}")

Starting data collection for 265 articles...
Fetching batch 1/11...
Collected 25 articles so far...
Fetching batch 2/11...
Collected 25 articles so far...
Fetching batch 2/11...
Collected 50 articles so far...
Fetching batch 3/11...
Collected 50 articles so far...
Fetching batch 3/11...
Collected 75 articles so far...
Fetching batch 4/11...
Collected 75 articles so far...
Fetching batch 4/11...
Collected 100 articles so far...
Fetching batch 5/11...
Collected 100 articles so far...
Fetching batch 5/11...
Collected 125 articles so far...
Fetching batch 6/11...
Collected 125 articles so far...
Fetching batch 6/11...
Collected 150 articles so far...
Fetching batch 7/11...
Collected 150 articles so far...
Fetching batch 7/11...
Collected 175 articles so far...
Fetching batch 8/11...
Collected 175 articles so far...
Fetching batch 8/11...


___________

In [11]:
# Criar DataFrame
dfScopus = pd.DataFrame(all_articles)
dfScopus.iloc[[0,-1]] # Exibe o DataFrame


,Title,Authors,Journal,DOI,Cited by,Affiliations,Abstract,Keywords,Index Keywords,Publisher,Conference Name,Conference Date,Language,Source,EID,ISSN,ISBN,PubMed ID,Funding Text
0,From disclosure to discrepancy: How open gover...,Hu J.,Government Information Quarterly,10.1016/j.giq.2025.102085,0,"[{'@_fa': 'true', 'affiliation-url': 'https://...",None,None,None,None,None,None,None,None,2-s2.0-105020574618,None,None,None,None
264,Towards ecosystems based on open data as a ser...,Gama K.,Iceis 2014 Proceedings of the 16th Internation...,None,15,"[{'@_fa': 'true', 'affiliation-url': 'https://...",None,None,None,None,None,None,None,None,2-s2.0-84902354877,None,None,None,None


In [15]:
dfScopus.columns

Index(['Title', 'Authors', 'Journal', 'DOI', 'Cited by', 'Affiliations',
       'Abstract', 'Keywords', 'Index Keywords', 'Publisher',
       'Conference Name', 'Conference Date', 'Language', 'Source', 'EID',
       'ISSN', 'ISBN', 'PubMed ID', 'Funding Text'],
      dtype='object')

In [16]:
dfScopus.Abstract

0      None
1      None
2      None
3      None
4      None
       ... 
260    None
261    None
262    None
263    None
264    None
Name: Abstract, Length: 265, dtype: object

In [12]:
dfScopus.Affiliations[0]

[{'@_fa': 'true',
  'affiliation-url': 'https://api.elsevier.com/content/affiliation/affiliation_id/60017605',
  'afid': '60017605',
  'affilname': 'Fuzhou University',
  'affiliation-city': 'Fuzhou',
  'affiliation-country': 'China'},
 {'@_fa': 'true',
  'affiliation-url': 'https://api.elsevier.com/content/affiliation/affiliation_id/60004630',
  'afid': '60004630',
  'affilname': 'Fujian Agriculture and Forestry University',
  'affiliation-city': 'Fuzhou',
  'affiliation-country': 'China'},
 {'@_fa': 'true',
  'affiliation-url': 'https://api.elsevier.com/content/affiliation/affiliation_id/60017431',
  'afid': '60017431',
  'affilname': 'Zhejiang University of Science and Technology',
  'affiliation-city': 'Hangzhou',
  'affiliation-country': 'China'}]

In [13]:
# Reset index to ensure it's preserved as a column
dfAffiliations = dfScopus[['Affiliations']].copy()
dfAffiliations['original_index'] = dfScopus.index

# Explode the affiliations while preserving the original index
dfAffiliations_exploded = pd.json_normalize(
    dfAffiliations.to_dict('records'),
    record_path='Affiliations',
    meta=['original_index'],
    errors='ignore'  # Skips over missing keys
)

In [14]:
dfAffiliations_exploded.iloc[[0,-1]]

,@_fa,affiliation-url,afid,affilname,affiliation-city,affiliation-country,original_index
0,true,https://api.elsevier.com/content/affiliation/a...,60017605,Fuzhou University,Fuzhou,China,0
522,true,https://api.elsevier.com/content/affiliation/a...,60031482,Universidade Federal de Pernambuco,Recife,Brazil,264


In [15]:

# Salvar o DataFrame em um arquivo CSV
dfScopus.to_csv(r'./data/dfScopus_bibliometric.csv', index=False)

# Analyze Key Bibliometric Indicators

### Summary of the Structure

#### **How Many?**
Explore counts and distributions of articles, authors, and citations to measure research productivity and impact.

- **Total Articles**: Count the number of articles in the dataset.
- **Top Authors by Productivity**: Use `Authors` to identify the most prolific authors.
- **Top Cited Articles**: Sort by `Cited by` to find the articles with the highest impact.
- **Most Frequent Document Types**: Analyze `Document Type` to see the distribution of different publication types (e.g., articles, reviews, conference papers).
- **Open Access vs. Closed Access**: Count and compare documents marked as `Open Access` vs. non-open access.

#### **When?**
Analyze temporal aspects to see trends over time in publication and citations.

- **Publications per Year**: Use `Year` to show trends in publications.
- **Citations per Year**: Sum `Cited by` by year to see citation trends.
- **Trends in Open Access Publications**: Track the increase in `Open Access` publications over time.
- **Publication Stage Trends**: Analyze `Publication Stage` (e.g., "Final" vs. "In Press") by year to understand publishing workflows.

#### **Where?**
Focus on sources, affiliations, and publication venues to understand where research is concentrated.

- **Top Journals by Number of Articles**: Use `Source title` to identify the most frequent publication venues.
- **Top Affiliations**: Analyze `Affiliations` to see which institutions are most active.
- **Geographic Distribution of Research**: If `Conference location` has country or region data, use it to map out geographic concentrations.
- **Frequent Conference Venues**: Use `Conference name` and `Conference location` to identify popular conferences in the field.
- **Top Publishers**: Count occurrences in `Publisher` to identify the main publishers in this area of research.

#### **What?**
Analyze topics, keywords, research focus, and funding patterns.

- **Top Keywords**: Use `Author Keywords` and `Index Keywords` to find common research topics.
- **Frequent Chemicals/CAS Numbers**: If relevant to your field, analyze `Chemicals/CAS` to identify frequently studied chemicals.
- **Common Funding Sources**: Use `Funding Details` or `Funding Texts` to identify major funding bodies or grants.
- **Abstract Word Cloud**: Generate a word cloud from `Abstract` text to visually identify common themes.
- **Research Sponsors**: Analyze `Sponsors` to see which organizations support this research.
- **Research on Molecular Sequences**: Count unique `Molecular Sequence Numbers` if applicable to study trends in sequence-based research.
- **Language of Publication**: Count occurrences in `Language of Original Document` to understand the language distribution in the research field.


## Part 1: How Many?
In this section, we analyze counts, such as the number of articles, citations, authors, and other similar metrics.

In [ ]:
# Total number of articles
total_articles = df.shape[0]
print(f"Total number of articles: {total_articles}")

In [ ]:
# Most Frequently Cited Articles
top_cited_articles = df[['Title', 'Cited by']].sort_values(by='Cited by', ascending=False).head(10)
print("Top 10 Most Cited Articles:")
print(top_cited_articles)

In [ ]:
# Most Productive Authors
author_counts = df['Authors'].str.split(', ').explode().value_counts().head(10)
print("Top 10 Most Productive Authors:")
print(author_counts)

In [ ]:
# Open Access vs. Closed Access counts
open_access_counts = df['Open Access'].value_counts()
print("Open Access vs. Closed Access:")
print(open_access_counts)

In [ ]:
# Plotting the bar chart
plt.figure(figsize=(8, 6))
open_access_counts.plot(kind='bar')

## Part 2: When?
This part will focus on temporal aspects, analyzing trends over time. For instance, it may examine how many articles were published each year.

In [ ]:
df['Year']

In [ ]:
# Yearly publication trends
yearly_publications = df['Year'].value_counts().sort_index()
print("Yearly Publication Trends:")
print(yearly_publications)


In [ ]:
# Plot the number of publications per year
plt.figure(figsize=(10, 6))
ax = yearly_publications.plot(kind='bar', title='Number of Publications per Year')
plt.xlabel("Year")
plt.ylabel("Number of Publications")

# Add labels on top of each bar
ax.bar_label(ax.containers[0])

plt.show()

In [ ]:
# Sum citations by year
citations_per_year = df.groupby('Year')['Cited by'].sum()

# Plot citations per year
plt.figure(figsize=(10, 6))
citations_per_year.plot(kind='line', marker='o', color='green', title='Citations per Year')
plt.xlabel("Year")
plt.ylabel("Total Citations")
plt.grid(True)
plt.show()

In [ ]:
# Display unique values in 'Open Access' column to check for inconsistencies
print(df['Open Access'].unique())

In [ ]:
# Create a binary column to indicate open access status
df['Is Open Access'] = df['Open Access'].str.contains("all open access", na=False)

# Group by 'Year' and 'Is Open Access', then count the number of documents
documents_by_access = df.groupby(['Year', 'Is Open Access']).size().unstack(fill_value=0)

# Plot quantity of documents per year for open and non-open access
plt.figure(figsize=(10, 6))
documents_by_access.plot(kind='line', marker='o', title='Number of Documents per Year: Open Access vs Non-Open Access')
plt.xlabel("Year")
plt.ylabel("Number of Documents")
plt.legend(["Non-Open Access", "Open Access"], title="Access Type")
plt.show()


In [ ]:
# Create a binary column to indicate open access status
df['Is Open Access'] = df['Open Access'].str.contains("all open access", na=False)

# Group by 'Year' and 'Is Open Access', then sum citations
citations_by_access = df.groupby(['Year', 'Is Open Access'])['Cited by'].sum().unstack(fill_value=0)

# Plot citations per year for open and non-open access
plt.figure(figsize=(10, 6))
citations_by_access.plot(kind='line', marker='o', title='Citations per Year: Open Access vs Non-Open Access')
plt.xlabel("Year")
plt.ylabel("Total Citations")
plt.legend(["Non-Open Access", "Open Access"], title="Access Type")
plt.show()


## Part 3: Where?
This section will examine where the research has been published, looking into journals, conferences, and affiliations.

In [ ]:
# Most Cited Journals
journal_citations = df.groupby('Source title')['Cited by'].sum().sort_values(ascending=False).head(10)
print("Top 10 Most Cited Journals:")
print(journal_citations)

In [ ]:
# Plot of Most Cited Journals
plt.figure(figsize=(10, 6))
ax = journal_citations.plot(kind='bar', title='Top Cited Journals')
plt.xlabel("Journal")
plt.ylabel("Total Citations")

# Add labels directly on top of the bars
ax.bar_label(ax.containers[0])

plt.show()


## Part 4: What?
In this final section, you will analyze the topics and themes within the research, such as keywords, abstract content, and funding sources.

In [ ]:
# Most Frequent Keywords
df['Document Type'] = df['Document Type'].fillna('')
all_keywords = df['Document Type'].str.split('; ').explode().value_counts().head(10)
print("Type of Documents:")
print(all_keywords)

# Plot top keywords
plt.figure(figsize=(10, 6))
all_keywords.plot(kind='bar', title='Type of Documents')
plt.xlabel("Documents")
plt.ylabel("Frequency")
plt.show()

In [ ]:
# Most Frequent Keywords
df['Author Keywords'] = df['Author Keywords'].fillna('')
all_keywords = df['Author Keywords'].str.split('; ').explode().value_counts().head(10)
print("Top 10 Author Keywords:")
print(all_keywords)

# Plot top keywords
plt.figure(figsize=(10, 6))
all_keywords.plot(kind='bar', title='Top Author Keywords')
plt.xlabel("Keyword")
plt.ylabel("Frequency")
plt.show()
